# RAG Piplines - Data Ingestion to Vector DB pipeline

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
def process_all_pdf(pdf_directory):

  all_documents = []
  pdf_dir = Path(pdf_directory) # get the directory of 

  pdf_files = list(pdf_dir.glob("**/*.pdf"))

  print(f"Found {len(pdf_files)} PDF Files to process")
  
  for pdf_file in pdf_files:
    print(f"\nProcessing: {pdf_file}")
    try:
      loader = PyPDFLoader(pdf_file)
      documents = loader.load()

      for doc in documents:
        doc.metadata['source_file'] = pdf_file.name
        doc.metadata['name'] = 'pdf'

      all_documents.extend(documents)
      print(f' loader{len(documents)} pages')
    
    except Exception as e:
      print(f" Error: {e}")
  print(f"Total Documents loaded: {len(all_documents)}")
  return all_documents


In [6]:
all_pdf_documents = process_all_pdf('../data')

Found 3 PDF Files to process

Processing: ..\data\text_files\Pdf\mars_lithograph.pdf
 loader2 pages

Processing: ..\data\text_files\Pdf\neptune_lithograph_h.pdf
 loader2 pages

Processing: ..\data\text_files\Pdf\saturn-lithograph.pdf
 loader2 pages
Total Documents loaded: 6


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2014-04-02T07:35:57-05:00', 'moddate': '2014-04-02T07:50:52-05:00', 'source': '..\\data\\text_files\\Pdf\\mars_lithograph.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'mars_lithograph.pdf', 'name': 'pdf'}, page_content='National Aeronautics and Space Administration\nNational Aeronautics and Space Administration\n300,000,000 900,000,000 1,500,000,000 2,100,000,000 2,700,000,000 3,300,000,000 3,900,000,000 4,500,000,000 5,100,000,000 5,700,000,000 kilometers0\nMars\nwww.nasa.gov'),
 Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2014-04-02T07:35:57-05:00', 'moddate': '2014-04-02T07:50:52-05:00', 'source': '..\\data\\text_files\\Pdf\\mars_lithograph.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'source_file': 'mars_lithograph.pdf', 'name': 'pdf'}, page_content='Tho

# Text Splitting get into chunks

In [8]:
def split_documents(document, chunk_size=1000,chunk_overlap=200):
  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap,
    length_function = len,
    separators = ['\n\n','\n',' ','']
  )
  split_docs = text_splitter.split_documents(document)
  print(f'split {len(document)} documents into {len(split_docs)} chunks')

  if split_docs:
    print(f"Example chunks")
    print(f"Content: {split_docs[0].page_content[:200]}")
    print(f"Metadata: {split_docs[0].metadata}")
  return split_docs

In [9]:
chunks = split_documents(all_pdf_documents)
chunks

split 6 documents into 29 chunks
Example chunks
Content: National Aeronautics and Space Administration
National Aeronautics and Space Administration
300,000,000 900,000,000 1,500,000,000 2,100,000,000 2,700,000,000 3,300,000,000 3,900,000,000 4,500,000,000 
Metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2014-04-02T07:35:57-05:00', 'moddate': '2014-04-02T07:50:52-05:00', 'source': '..\\data\\text_files\\Pdf\\mars_lithograph.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'mars_lithograph.pdf', 'name': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2014-04-02T07:35:57-05:00', 'moddate': '2014-04-02T07:50:52-05:00', 'source': '..\\data\\text_files\\Pdf\\mars_lithograph.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'mars_lithograph.pdf', 'name': 'pdf'}, page_content='National Aeronautics and Space Administration\nNational Aeronautics and Space Administration\n300,000,000 900,000,000 1,500,000,000 2,100,000,000 2,700,000,000 3,300,000,000 3,900,000,000 4,500,000,000 5,100,000,000 5,700,000,000 kilometers0\nMars\nwww.nasa.gov'),
 Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2014-04-02T07:35:57-05:00', 'moddate': '2014-04-02T07:50:52-05:00', 'source': '..\\data\\text_files\\Pdf\\mars_lithograph.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'source_file': 'mars_lithograph.pdf', 'name': 'pdf'}, page_content='Tho

# Embedding and Vector Store

In [10]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple

c:\Users\Admin\Pojects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class EmbeddingManager:
  """
  Initialize the embedding manager

  Args:
        model_name = HuggingFace model name for sentence embeddings
  """
  def __init__(self, model_name : str = 'all-MiniLM-L6-v2'):
    self.model_name = model_name
    self.model = None
    self.load_model()

  def load_model(self):
    """Load the SentenceTransformer model"""
    try:
      print(f"Loading the embedding model: {self.model_name}")
      self.model = SentenceTransformer(self.model_name)
      print(f"Model Loaded Sucessfully. Embedding Dimension: {self.model.get_embedding_dimension()}")

    except Exception as e:
      print(f"Error loading model {self.model_name}:{e}")
      raise

  def embeddings(self, texts: list[str]) -> np.ndarray:
    """
    Generate embeddings for the list of texts

    Args:
          texts : List of text strings to convert into embeddings
    
    Returns:
          Numpy array of embeddings with the shape (len(texts), embedding_dim)
    """
    if not self.model:
      raise ValueError('Model is not loaded')

    print(f'Generate Embeddings for {len(texts)} texts...')
    embeddings = self.model.encode(texts, show_progress_bar=True)
    print(f'Generated embeddings with shape: {embeddings.shape}')
    return embeddings




In [12]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading the embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2147.77it/s]


Model Loaded Sucessfully. Embedding Dimension: 384


## Vector Store

In [24]:
class VectorStore:
  """Manages document embeddings in a chromaDB vectore store"""
  def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = '..data/Vector_store'):
    """
    Initialize the vector store

    Args:
    collection_name: name of the chromaDB collection
    presist_directory: Directory to presist the vector store
    """

    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self.initialize_store()


  def initialize_store(self):
    """Initialize ChromaDB client and collection"""

    try:
      # Create Persistent ChromaDB client
      os.makedirs(self.persist_directory, exist_ok = True)
      self.client = chromadb.PersistentClient(path=self.persist_directory)

      #Get or Create Collection (where we want to store the vectors)
      self.collection = self.client.get_or_create_collection(
        name=self.collection_name,
        metadata = {'description': 'PDF document embeddings for RAG'}
      )
      print(f'Vector store Initialized. collection: {self.collection_name}')
      print(f'Exsisting documents in collection: {self.collection}')

    except Exception as e:
      print(f'Error initializing Vector Store: {e}')
      raise

  def add_documents(self, documents: List[Any], embbeddings: np.ndarray):
    """
    Add documents and their embeddings to the Vector Store

    Args:
        documents: List of Langchain documents
        embeddings: corresponding embeddings for the documents
    """
    if (len(documents)) != (len(embbeddings)):
      raise ValueError('Number of docuemnts must match number of embeddings')

    print(f'Adding {len(documents)} documents to vector store...')

    #prepare data for ChromaDB

    ids=[]
    metadatas=[]
    documents_text=[]
    embeddings_list=[]

    for i, (doc, embbed) in enumerate(zip(documents,embbeddings)):
      doc_id = f"{uuid.uuid4().hex[:8]}_{i}"
      ids.append(doc_id)

      # prepare metadata
      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_length'] = len(doc.page_content)
      metadatas.append(metadata)

      documents_text.append(doc.page_content)

      embeddings_list.append(embbed.tolist())

    try:
      self.collection.add(
        ids=ids,
        embeddings=embeddings_list,
        metadatas=metadatas,
        documents=documents_text
      )

      print(f'Sucessfully added {len(documents)} documents to vector store')
      print(f'Total Documents in collection: {self.collection.count()}')

    except Exception as e:
      print(f'Error adding documents to vector store {e}')
      raise

vectorstore = VectorStore()
vectorstore

Vector store Initialized. collection: pdf_documents
Exsisting documents in collection: Collection(name=pdf_documents)


In [25]:
vectorstore.initialize_store()

Vector store Initialized. collection: pdf_documents
Exsisting documents in collection: Collection(name=pdf_documents)


In [93]:
f"{uuid.uuid4().hex[:8]}_{0}"


'67e60674_0'